# 03 Feature Baselines

Stage 6 trains traditional machine-learning baselines on compact engineered epoch-level features. The setup cell imports the reusable feature and evaluation helpers, defines repository paths, and checks whether XGBoost is available. Model selection uses the training split only through participant-level cross-validation. Validation is used once for interim evaluation and diagnostics. The held-out test split is saved as features for later final evaluation, but it is not used here for training, tuning, permutation importance, or conclusions.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, f1_score
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

repo_root = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
data_processed_dir = repo_root / "data" / "processed"
raw_dir = repo_root / "data" / "raw"
epoch_index_path = repo_root / "data" / "interim" / "epoch_index.csv"
split_assignments_path = repo_root / "data" / "interim" / "split_assignments.csv"
results_dir = repo_root / "results"
results_dir.mkdir(parents=True, exist_ok=True)

import sys
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.evaluate import evaluate_predictions
from src.features import FEATURE_ID_COLUMNS, build_feature_table, save_feature_tables
from src.preprocessing import TARGET_SLEEP_STAGE_LABELS

## Build Or Load Features

This section loads existing engineered feature CSVs from `data/processed/` when they are already present. If they are missing, it builds them from `data/raw/`, `data/interim/epoch_index.csv`, and `data/interim/split_assignments.csv` using `src.features.build_feature_table`, then saves separate train, validation, and test CSVs. The expected output is a small split summary with epoch and participant counts. The test feature table is created only as a saved artifact for future final evaluation.

In [ ]:
feature_paths = {
    "train": data_processed_dir / "features_train.csv",
    "validation": data_processed_dir / "features_val.csv",
    "test": data_processed_dir / "features_test.csv",
}

if all(path.exists() for path in feature_paths.values()):
    train_df = pd.read_csv(feature_paths["train"], dtype={"participant_id": str})
    val_df = pd.read_csv(feature_paths["validation"], dtype={"participant_id": str})
    test_df = pd.read_csv(feature_paths["test"], dtype={"participant_id": str})
else:
    features = build_feature_table(
        raw_dir=raw_dir,
        epoch_index_path=epoch_index_path,
        split_assignments_path=split_assignments_path,
    )
    train_df = features[features["split"] == "train"].reset_index(drop=True)
    val_df = features[features["split"] == "validation"].reset_index(drop=True)
    test_df = features[features["split"] == "test"].reset_index(drop=True)
    save_feature_tables(train_df, val_df, test_df, data_processed_dir)

display(pd.DataFrame({
    "split": ["train", "validation", "test"],
    "n_epochs": [len(train_df), len(val_df), len(test_df)],
    "n_participants": [
        train_df["participant_id"].nunique(),
        val_df["participant_id"].nunique(),
        test_df["participant_id"].nunique(),
    ],
}))

## Validation Setup

This section separates identifier columns from model features, creates `X`/`y` objects for train and validation, and configures `GroupKFold` so cross-validation folds are split by `participant_id`. It also defines macro-F1 scorers and a helper that records validation metrics and confusion matrices in a consistent format. The expected output is no displayed table; later sections reuse these objects.

In [ ]:
feature_columns = [column for column in train_df.columns if column not in FEATURE_ID_COLUMNS]
X_train = train_df[feature_columns]
y_train = train_df["label"]
groups_train = train_df["participant_id"]

X_val = val_df[feature_columns]
y_val = val_df["label"]

n_splits = min(5, groups_train.nunique())
if n_splits < 2:
    raise ValueError("Participant-level cross-validation needs at least two training participants.")

cv = GroupKFold(n_splits=n_splits)
macro_f1 = make_scorer(f1_score, average="macro", labels=TARGET_SLEEP_STAGE_LABELS, zero_division=0)
macro_f1_unlabeled = make_scorer(f1_score, average="macro", zero_division=0)
metrics_rows = []
confusion_matrices = {}

def record_validation_result(model_name, predictions):
    metrics, matrix = evaluate_predictions(
        y_val,
        predictions,
        model_name=model_name,
        split="validation",
    )
    metrics_rows.append(metrics)
    confusion_matrices[model_name] = matrix
    return metrics, matrix

## Majority Class Baseline

This sanity-check model always predicts the most frequent training label. It is fit on the training feature table and evaluated once on the validation feature table. The expected outputs are one validation metrics row and a labeled confusion matrix, which establish the minimum useful benchmark for later models.

In [ ]:
majority_model = DummyClassifier(strategy="most_frequent")
majority_model.fit(X_train, y_train)
majority_predictions = majority_model.predict(X_val)
majority_metrics, majority_confusion = record_validation_result(
    "majority_class",
    majority_predictions,
)
display(pd.DataFrame([majority_metrics]))
display(majority_confusion)

## Elastic-Net Multinomial Logistic Regression

This section fits a scikit-learn pipeline with median imputation, standardization, and balanced multinomial logistic regression using elastic-net regularization. `GridSearchCV` tunes `C` and `l1_ratio` with participant-level cross-validation on the training split only; the selected pipeline is then evaluated once on validation. Expected outputs are the top CV settings, a validation metrics row, and a confusion matrix.

In [ ]:
logistic_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        (
            "model",
            LogisticRegression(
                penalty="elasticnet",
                solver="saga",
                multi_class="multinomial",
                class_weight="balanced",
                max_iter=5000,
                random_state=42,
            ),
        ),
    ]
)

logistic_search = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid={
        "model__C": [0.01, 0.1, 1.0, 10.0],
        "model__l1_ratio": [0.0, 0.5, 1.0],
    },
    scoring=macro_f1,
    cv=cv,
    n_jobs=-1,
    refit=True,
)
logistic_search.fit(X_train, y_train, groups=groups_train)
logistic_predictions = logistic_search.predict(X_val)
logistic_metrics, logistic_confusion = record_validation_result(
    "logistic_elasticnet",
    logistic_predictions,
)
display(pd.DataFrame(logistic_search.cv_results_).sort_values("rank_test_score").head(10))
display(pd.DataFrame([logistic_metrics]))
display(logistic_confusion)

## XGBoost Baseline

This section trains XGBoost on all engineered features when the package is installed. Labels are encoded for XGBoost, and a modest grid over tree depth, learning rate, sampling, and regularization is tuned with participant-level training CV only. The selected model is evaluated once on validation, then validation-set permutation importance is computed as a diagnostic. Expected outputs are the top CV settings, validation metrics, a confusion matrix, and the most important features by permutation importance.

In [ ]:
if XGBClassifier is None:
    print("xgboost is not installed; install it to run the XGBoost baseline.")
    xgb_search = None
    xgb_importance = pd.DataFrame()
else:
    label_encoder = LabelEncoder()
    y_train_encoded = label_encoder.fit_transform(y_train)
    y_val_encoded = label_encoder.transform(y_val)

    xgb_model = XGBClassifier(
        objective="multi:softprob",
        num_class=len(label_encoder.classes_),
        eval_metric="mlogloss",
        tree_method="hist",
        random_state=42,
    )
    xgb_search = GridSearchCV(
        estimator=xgb_model,
        param_grid={
            "max_depth": [2, 3, 4],
            "learning_rate": [0.03, 0.1],
            "subsample": [0.8, 1.0],
            "colsample_bytree": [0.8, 1.0],
            "reg_lambda": [1.0, 5.0],
            "min_child_weight": [1, 5],
        },
        scoring=macro_f1_unlabeled,
        cv=cv,
        n_jobs=-1,
        refit=True,
    )
    xgb_search.fit(X_train, y_train_encoded, groups=groups_train)
    xgb_predictions = label_encoder.inverse_transform(xgb_search.predict(X_val))
    xgb_metrics, xgb_confusion = record_validation_result(
        "xgboost_all_features",
        xgb_predictions,
    )

    validation_importance = permutation_importance(
        xgb_search.best_estimator_,
        X_val,
        y_val_encoded,
        scoring=macro_f1_unlabeled,
        n_repeats=10,
        random_state=42,
        n_jobs=-1,
    )
    xgb_importance = pd.DataFrame(
        {
            "feature": feature_columns,
            "importance_mean": validation_importance.importances_mean,
            "importance_std": validation_importance.importances_std,
        }
    ).sort_values("importance_mean", ascending=False)

    display(pd.DataFrame(xgb_search.cv_results_).sort_values("rank_test_score").head(10))
    display(pd.DataFrame([xgb_metrics]))
    display(xgb_confusion)
    display(xgb_importance.head(30))

## Interim Validation Summary

This final section combines validation metrics from the baselines, saves them to `results/stage6_validation_metrics.csv`, and displays each confusion matrix. If XGBoost ran successfully, it also saves validation-set permutation importance to `results/stage6_xgboost_validation_permutation_importance.csv`. These outputs are interim validation diagnostics, not final test-set comparisons.

In [ ]:
metrics_df = pd.DataFrame(metrics_rows)
display(metrics_df.sort_values("macro_f1", ascending=False))

metrics_output_path = results_dir / "stage6_validation_metrics.csv"
metrics_df.to_csv(metrics_output_path, index=False)
print(f"Saved validation metrics to {metrics_output_path}")

for model_name, matrix in confusion_matrices.items():
    display(model_name)
    display(matrix)

if not xgb_importance.empty:
    importance_output_path = results_dir / "stage6_xgboost_validation_permutation_importance.csv"
    xgb_importance.to_csv(importance_output_path, index=False)
    print(f"Saved validation-set permutation importance to {importance_output_path}")